# Кластеризация и снижение размерности

## Содержание

0. Подготовка среды и подключение Weights & Biases
1. Данные и постановка задачи без учителя
2. Метод k-средних (k-means)
3. Другие методы: иерархическая кластеризация и DBSCAN
4. Снижение размерности: метод главных компонент (PCA)
5. Практикум

До сих пор у нас была **разметка**: мы предсказывали известную целевую переменную. Теперь переходим к **обучению без учителя** — размечать данные нужно самим. Сначала группируем объекты на похожие (кластеризация), затем учимся сжимать много признаков в несколько главных направлений (снижение размерности), чтобы данные было проще увидеть и описать.

Демонстрационный пример — датасет о химическом составе вин (три сорта). Практикум (раздел 5) вы выполняете на **датасете своего проекта**. Выводы после каждого блока записывайте **в ячейке markdown прямо в блокноте**.

## 0. Подготовка среды и подключение Weights & Biases

К библиотекам прошлых занятий добавляется `scipy` для дендрограммы. **Примечание.** Если с W&B проблемы, замените `wandb.init(...)` на `wandb.init(mode="disabled")`.

In [ ]:
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score
from scipy.cluster.hierarchy import linkage, dendrogram

import wandb

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (8, 5)
RANDOM_STATE = 42
wandb.login()

: 

## 1. Данные и постановка задачи без учителя

Датасет `load_wine` содержит результаты химического анализа 178 образцов вина; у каждого — 13 числовых признаков (содержание алкоголя, магний, фенолы и т.д.). Известен и истинный сорт (их три), но в обучении без учителя мы эту метку **не используем** — она понадобится только в конце, чтобы проверить, насколько кластеры совпали с реальными сортами.

**Постановка.** Дана матрица объектов $X$ без целевой переменной. Нужно разбить объекты на группы так, чтобы внутри группы объекты были похожи, а между группами — отличались.

**Важно:** кластеризация и снижение размерности чувствительны к масштабу признаков, поэтому признаки стандартизуем.

In [ ]:
wine = load_wine(as_frame=True)
X = wine.data
y_true = wine.target  # только для проверки в конце, в кластеризации НЕ используется

print("Размер таблицы:", X.shape)
print("Истинные классы:", np.bincount(y_true))
X.describe().loc[['mean', 'std', 'min', 'max']].round(2)

In [ ]:
# Масштаб признаков очень разный — без стандартизации k-means и PCA «поедут»
scaler = StandardScaler()
Xs = scaler.fit_transform(X)
print("После стандартизации: mean ≈ 0, std ≈ 1 для каждого признака")

## 2. Метод k-средних (k-means)

### Математический аппарат

Алгоритм задаёт $k$ центров $\mu_1, \dots, \mu_k$ и минимизирует **внутрикластерную сумму квадратов** (WCSS, она же инерция):

$$J = \sum_{i=1}^{n} \lVert x_i - \mu_{c(i)} \rVert^2 \to \min,$$

где $c(i)$ — номер кластера объекта $x_i$. Минимум ищут итеративно (**алгоритм Ллойда**): назначить каждый объект ближайшему центру → пересчитать центры как средние → повторять. Каждый шаг не увеличивает $J$, поэтому процесс сходится (к локальному минимуму).

**Как выбрать $k$.** Инерция $J$ всегда падает с ростом $k$ (при $k=n$ она равна нулю), поэтому «минимум $J$» не годится. Используют:

- **метод локтя** — искать излом на графике $J(k)$;
- **силуэт** $s = \dfrac{b - a}{\max(a, b)}$, где $a$ — среднее расстояние до объектов своего кластера, $b$ — до ближайшего чужого. $s \in [-1, 1]$, больше — лучше.

**Примечание.** Кластеры k-means по построению круглые и сравнимые по размеру; на вытянутых или плотностных структурах метод работает плохо — тогда смотрите раздел 3.

In [ ]:
km = KMeans(n_clusters=3, n_init=10, random_state=RANDOM_STATE)
labels = km.fit_predict(Xs)

print("Инерция:", round(km.inertia_, 1))
print("Силуэт:", round(silhouette_score(Xs, labels), 3))
print("Совпадение с истинными сортами (ARI):", round(adjusted_rand_score(y_true, labels), 3))

# сопоставим кластеры и истинные классы
print("\nТаблица соответствия (кластер × истинный сорт):")
print(pd.crosstab(labels, y_true))

In [ ]:
# Выбор k: инерция (локоть) и силуэт
rows = []
for k in [2, 3, 4, 5, 6]:
    m = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(Xs)
    rows.append((k, m.inertia_, silhouette_score(Xs, m.labels_)))

k_table = pd.DataFrame(rows, columns=['k', 'inertia', 'silhouette'])
print(k_table.round(3).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(k_table['k'], k_table['inertia'], marker='o')
axes[0].set_xlabel('k'); axes[0].set_ylabel('инерция'); axes[0].set_title('Метод локтя')
axes[1].plot(k_table['k'], k_table['silhouette'], marker='o')
axes[1].set_xlabel('k'); axes[1].set_ylabel('силуэт'); axes[1].set_title('Силуэт по k')
plt.tight_layout(); plt.show()

**Задача 1.**
Какому $k$ соответствует излом на графике инерции и максимум силуэта? Сколько образцов оказалось «не в своём» кластере по сравнению с истинными сортами? Сделайте вывод — в «Выводе».

**Вывод:**

_..._

## 3. Другие методы: иерархическая кластеризация и DBSCAN

### Математический аппарат

- **Иерархическая (агломеративная) кластеризация** последовательно объединяет ближайшие объекты/группы. Результат удобно изображать **дендрограммой**; число кластеров выбирают, «разрезая» дерево на нужной высоте. Расстояние между кластерами задаётся связью (ward, average, complete).
- **DBSCAN** — плотностный метод: кластеры — это области высокой плотности, разделённые областями низкой. Ему нужно два параметра: радиус `eps` и минимум соседей `min_samples`. Он сам определяет число кластеров и **помечает выбросы** меткой `-1`, но плохо работает, когда плотность кластеров сильно разная.

**Смысл:** у разных методов разные представления о «похожести»: k-means делит по расстоянию до центра, иерархия — по цепочке слияний, DBSCAN — по плотности. Выбор метода зависит от формы кластеров.

In [ ]:
# Агломеративная кластеризация + дендрограмма
Z = linkage(Xs, method='ward')
plt.figure(figsize=(10, 4))
dendrogram(Z, truncate_mode='lastp', p=20, show_leaf_counts=True)
plt.title('Дендрограмма (ward)')
plt.xlabel('кластеры'); plt.ylabel('расстояние слияния')
plt.show()

ag = AgglomerativeClustering(n_clusters=3).fit(Xs)
print("Агломеративная (3): ARI =", round(adjusted_rand_score(y_true, ag.labels_), 3),
      "| силуэт =", round(silhouette_score(Xs, ag.labels_), 3))

In [ ]:
# DBSCAN: перебор eps (min_samples фиксирован)
for eps in [1.5, 2.0, 2.5]:
    db = DBSCAN(eps=eps, min_samples=5).fit(Xs)
    labs = db.labels_
    n_clusters = len(set(labs)) - (1 if -1 in labs else 0)
    noise = int((labs == -1).sum())
    ari = adjusted_rand_score(y_true, labs) if n_clusters > 1 else float('nan')
    print(f"eps={eps}: кластеров={n_clusters}, выбросов={noise}, ARI={ari:.3f}")

**Задача 2.**
Сравните k-means, агломеративную кластеризацию и DBSCAN по ARI и силуэту. Какой метод лучше совпал с истинными сортами и почему? Что DBSCAN считает выбросами? Ответ — в «Выводе».

**Вывод:**

_..._

## 4. Снижение размерности: метод главных компонент (PCA)

### Математический аппарат

Идея PCA — найти такие направления в пространстве признаков, вдоль которых данные разбросаны сильнее всего, и спроецировать объекты на несколько первых из них. Пусть признаки центрированы, а $C = \frac{1}{n}X^\top X$ — ковариационная матрица. Тогда главные компоненты — это **собственные векторы** $C$, а соответствующие собственные значения равны дисперсиям вдоль этих направлений:

$$C v_j = \lambda_j v_j, \qquad \lambda_1 \ge \lambda_2 \ge \dots$$

**Объяснённая дисперсия** доли $j$ — это $\lambda_j / \sum_i \lambda_i$. Оставляя первые 2 компоненты, мы теряем часть информации, но получаем плоскую картинку для визуализации.

*Смысл:* PCA не использует метки — это тоже обучение без учителя. Он полезен для визуализации, шумоподавления и ускорения последующих моделей, но новые признаки — это линейные комбинации исходных и их не так просто интерпретировать.

**Задача 3 (на вывод).**
Покажите, что первая главная компонента максимизирует дисперсию проекции $\mathrm{Var}(Xv)$ при $\lVert v \rVert = 1$, и что решение — собственный вектор ковариационной матрицы с наибольшим собственным значением.

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(Xs)

print("Объяснённая дисперсия по компонентам:", [round(v, 3) for v in pca.explained_variance_ratio_])
print("Суммарно двумя компонентами:", round(pca.explained_variance_ratio_.sum(), 3))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=labels, cmap='tab10', s=40)
axes[0].set_title('PCA: кластеры k-means')
axes[0].set_xlabel('Главная компонента 1'); axes[0].set_ylabel('Главная компонента 2')
axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=y_true, cmap='tab10', s=40)
axes[1].set_title('PCA: истинные сорта')
axes[1].set_xlabel('Главная компонента 1'); axes[1].set_ylabel('Главная компонента 2')
plt.tight_layout(); plt.show()

In [ ]:
# Логируем k-means в W&B
run = wandb.init(project="apml-course", name="kmeans-wine", config={"k": 3})
wandb.log({"inertia": km.inertia_, "silhouette": silhouette_score(Xs, labels)})
run.finish()

**Ваш вывод:**

_..._

## 5. Практикум

Задания выполняются на **датасете своего проекта**. Кластеризация не требует целевой переменной, поэтому берите числовые признаки (категориальные предварительно закодируйте one-hot). После каждого задания заполните ячейку **«Вывод»**.

### Задание 5.1. Подготовка признаков

Выберите набор признаков для кластеризации, обработайте пропуски и стандартизуйте. Объясните, почему для k-means и PCA масштабирование обязательно.

**Вывод:**

_..._

### Задание 5.2. k-means и выбор числа кластеров

Обучите k-means для нескольких значений $k$. Постройте графики инерции и силуэта и выберите разумное $k$.

**Вывод:**

_..._

### Задание 5.3. Интерпретация кластеров

Опишите полученные кластеры: их размеры и средние значения признаков. Какие группы объектов получились и как бы вы их назвали содержательно?

**Вывод:**

_..._

### Задание 5.4. Снижение размерности и визуализация

Примените PCA к своим данным. Сколько компонент нужно, чтобы объяснить ~80–90% дисперсии? Постройте объекты на первых двух компонентах и раскрасьте их по кластерам.

**Вывод:**

_..._

### Задание 5.5. Сравнение методов

Сравните k-means, агломеративную кластеризацию и DBSCAN на ваших данных по силуэту (и по числу найденных выбросов у DBSCAN). Какой метод дал более осмысленные группы?

**Резюме:**

_..._

---

## P.S. Как измерять качество кластеризации

У кластеризации нет «правильного ответа» как у классификации, поэтому используют два типа метрик.

- **Внутренние** (без разметки): **силуэт** (выше — лучше), индекс Калински–Харабаша, инерция. Они оценивают, насколько компактны и разделены кластеры.
- **Внешние** (если разметка есть, как в демо): **ARI**, **NMI** — насколько кластеры совпали с истинными классами. Нужны для проверки метода, но в реальной задаче метки обычно неизвестны.

**Главная ловушка.** Кластеры — это **не** классы. Алгоритм найдёт группы даже в случайном шуме, а найденная группа может не иметь бизнес-смысла. Поэтому кластеры всегда интерпретируют содержательно (профили, размеры) и проверяют на устойчивость (меняя seed, признаки, k) — а не принимают на веру только по цифре силуэта.

### Справочные материалы

- Кластеризация (scikit-learn): https://scikit-learn.org/stable/modules/clustering.html
- K-means: https://scikit-learn.org/stable/modules/clustering.html#k-means
- DBSCAN: https://scikit-learn.org/stable/modules/clustering.html#dbscan
- PCA: https://scikit-learn.org/stable/modules/decomposition.html#pca
- Метрики кластеризации: https://scikit-learn.org/stable/modules/clustering.html#clustering-performance-evaluation
- Интерактивная визуализация PCA: https://setosa.io/ev/principal-component-analysis/